In [26]:
from typing import List, Tuple, Optional

P = 71  # the prime modulus
A = 1  # coefficient of x in the curve equation
B = 28  # constant term in the curve equation


def legendre_symbol(a, p):
    """Return the Legendre symbol (a|p).  p must be an odd prime."""
    ls = pow(a % p, (p - 1) // 2, p)
    return -1 if ls == p - 1 else ls  # returns -1, 0 or 1

def tonelli_shanks(n, p):
    """
    Solve  x^2 ≡ n (mod p)  for prime p.
    Returns one solution x (the other is p-x) or None if no solution exists.
    """
    assert 0 <= n < p
    if n == 0:
        return 0
    if p == 2:
        return n

    # check that a solution exists
    if legendre_symbol(n, p) != 1:
        return None

    # factor p-1 = q * 2^s,  with q odd
    q = p - 1
    s = 0
    while q % 2 == 0:
        q //= 2
        s += 1

    # find a quadratic non‑residue z
    z = 2
    while legendre_symbol(z, p) != -1:
        z += 1

    # initialise the algorithm
    m = s
    c = pow(z, q, p)
    t = pow(n, q, p)
    r = pow(n, (q + 1) // 2, p)

    while t != 1:
        # find the smallest i (0 < i < m) with t^{2^i} = 1
        i = 1
        t2i = pow(t, 2, p)
        while i < m:
            if t2i == 1:
                break
            t2i = pow(t2i, 2, p)
            i += 1

        # update the variables
        b = pow(c, 1 << (m - i - 1), p)
        m = i
        c = pow(b, 2, p)
        t = (t * c) % p
        r = (r * b) % p

    return r  # one of the two square‑roots


def points_on_curve():
    """Return a list of all affine points (x, y) on the curve"""
    pts = []
    for x in range(P):
        rhs = (pow(x, 3, P) + A * x + B) % P
        y = tonelli_shanks(rhs, P)
        if y is None:
            continue  # no solution for this x
        # two solutions: y and -y (mod P); they may coincide when y == 0
        pts.append((x, y))
        if y != 0:
            pts.append((x, (-y) % P))
    return pts



affine_pts = points_on_curve()
# sort for nicer output
affine_pts.sort()
# add the point at infinity
all_pts = affine_pts + [("O", "O")]  # O denotes the neutral element

print(f"Elliptic curve  y^2 = x^3 + x + 28  over  Z_{P}")
print(f"Number of affine points : {len(affine_pts)}")
print(f"Number of points (incl. O): {len(all_pts)}\n")

print("Points (x, y) :")
for pt in affine_pts:
    print(f"({pt[0]:2d}, {pt[1]:2d})")
print("\nPoint at infinity : O")


Elliptic curve  y^2 = x^3 + x + 28  over  Z_71
Number of affine points : 71
Number of points (incl. O): 72

Points (x, y) :
( 1, 32)
( 1, 39)
( 2, 31)
( 2, 40)
( 3, 22)
( 3, 49)
( 4,  5)
( 4, 66)
( 5,  4)
( 5, 67)
( 6, 26)
( 6, 45)
(12,  8)
(12, 63)
(13, 26)
(13, 45)
(15,  9)
(15, 62)
(19, 27)
(19, 44)
(20,  5)
(20, 66)
(21,  3)
(21, 68)
(22, 30)
(22, 41)
(23, 19)
(23, 52)
(25, 22)
(25, 49)
(27,  0)
(31, 32)
(31, 39)
(33,  1)
(33, 70)
(34, 23)
(34, 48)
(35, 14)
(35, 57)
(36, 12)
(36, 59)
(37, 33)
(37, 38)
(39, 32)
(39, 39)
(41,  7)
(41, 64)
(43, 22)
(43, 49)
(47,  5)
(47, 66)
(48, 11)
(48, 60)
(49, 24)
(49, 47)
(52, 26)
(52, 45)
(53,  0)
(58, 27)
(58, 44)
(61, 15)
(61, 56)
(62,  0)
(63, 17)
(63, 54)
(65, 27)
(65, 44)
(66, 18)
(66, 53)
(69, 35)
(69, 36)

Point at infinity : O


In [27]:
P = 127  # prime modulus
A = 1  # coefficient of x
B = 26  # constant term


def inv_mod(a):
    return pow(a % P, P - 2, P)  # Fermat's little theorem


# The point at infinity is represented by the tuple (None, None)
O = (None, None)  # neutral element


def is_on_curve(Pt):
    """Check whether Pt = (x,y) satisfies the curve equation."""
    if Pt == O:
        return True
    x, y = Pt
    return (y * y - (x * x * x + A * x + B)) % P == 0


def point_neg(Pt):
    """Return the inverse of Pt:  -(x,y) = (x, -y)."""
    if Pt == O:
        return O
    x, y = Pt
    return (x, (-y) % P)


def point_add(P1, P2):
    """Add two points P1 and P2 (the usual chord and tangent law)."""
    # Handle the special cases involving the point at infinity
    if P1 == O:
        return P2
    if P2 == O:
        return P1

    x1, y1 = P1
    x2, y2 = P2

    # P1 == -P2, result is the point at infinity
    if x1 == x2 and (y1 + y2) % P == 0:
        return O

    if P1 != P2:
        # slope lambda = (y2 - y1) / (x2 - x1)
        lam = ((y2 - y1) * inv_mod(x2 - x1)) % P
    else:  # point doubling
        # lambda = (3 * x1^2 + A) / (2 * y1)
        lam = ((3 * x1 * x1 + A) * inv_mod(2 * y1)) % P

    x3 = (lam * lam - x1 - x2) % P
    y3 = (lam * (x1 - x3) - y1) % P
    return (x3, y3)


def scalar_mul(k, Pt):
    """Return k * Pt using the double and add method (k >= 0)."""
    if k % P == 0 or Pt == O:
        return O
    if k < 0:  # k * P = -(-k * P)
        return point_neg(scalar_mul(-k, Pt))

    result = O
    addend = Pt

    while k:
        if k & 1:
            result = point_add(result, addend)
        addend = point_add(addend, addend)  # double
        k >>= 1
    return result


P_a = (9, 16)
Q_a = (6, 11)
P_b = (11, 15)
P_c = (4, 27)

# (a)  (9,16) + (6,11)
R_a = point_add(P_a, Q_a)

# (b)  -(11,15)
R_b = point_neg(P_b)

# (c)  2(4,27)
R_c = scalar_mul(2, P_c)

print("Results on the curve y^2 = x^3 + x + 26 over Z_127")
print(f"(a)  (9,16) + (6,11) = {R_a}")
print(f"(b)  -(11,15)       = {R_b}")
print(f"(c)  2(4,27)       = {R_c}")

Results on the curve y^2 = x^3 + x + 26 over Z_127
(a)  (9,16) + (6,11) = (16, 57)
(b)  -(11,15)       = (11, 112)
(c)  2(4,27)       = (96, 120)


In [28]:
P = 4339
A = 193
B = 647

# base point Pt (order = 4378)
Pt = (719, 3538)

# public point Q = m * Pt  (the value of m is unknown)
Q = (3509, 334)

def point_decompress(comp):
    """
    comp = (x, parity)  where parity = 0 → even y,  parity = 1 → odd y
    Returns the full point (x, y) on the curve.
    """
    x, parity = comp
    rhs = (pow(x, 3, P) + A * x + B) % P
    y = tonelli_shanks(rhs, P)
    if y is None:
        raise ValueError(f"x = {x} is not on the curve")
    # choose the root with the required parity
    if (y % 2) != parity:
        y = P - y
    return (x, y)


def point_compress(Pt):
    """Return (x, parity) for a point Pt ≠ O."""
    x, y = Pt
    return (x, y % 2)

def find_private_key():
    """Bruteforce search"""
    for m in range(1, 4378):
        if scalar_mul(m, Pt) == Q:
            return m
    raise ValueError("private key not found")


ciphertext = [
    ((3103, 1), 1860),
    ((745, 1), 1308),
    ((2214, 0), 981),
    ((3210, 0), 3601),
    ((1222, 0), 3579),
    ((3643, 0), 2402),
    ((1449, 0), 1871),
    ((3450, 1), 584),
    ((556, 1), 3019),
    ((3945, 0), 148),
    ((468, 0), 4242),
    ((277, 0), 2557),
    ((1460, 0), 3434),
    ((711, 0), 1522),
    ((3034, 1), 3293),
    ((1565, 0), 848),
]

def decrypt(cipher, m):
    """Decrypt a list of ciphertext blocks."""
    plaintext_numbers = []
    for comp, c in cipher:
        # 1. recover k*P from the compressed point
        kP = point_decompress(comp)

        # 2. compute k*Q = m*(k·P)
        kQ = scalar_mul(m, kP)

        # 3. plaintext = c * (kQ.x)^-1  (mod p)
        x0 = kQ[0]
        inv_x0 = inv_mod(x0)
        x = (c * inv_x0) % P
        # the scheme works only with non‑zero plaintexts; 0 never occurs
        if x == 0:
            raise ValueError("decryption produced a zero")
        plaintext_numbers.append(x)
    return plaintext_numbers

def numbers_to_text(nums):
    """Map 1-26 → A-Z"""
    alphabet = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"
    return "".join(alphabet[n - 1] for n in nums)



# (a) private key
m = find_private_key()
print(f"Private key m = {m}")

# (b) decrypt
plain_nums = decrypt(ciphertext, m)
print("Plaintext numbers :", plain_nums)

# (c) convert to a readable phrase
phrase = numbers_to_text(plain_nums)
print("Plaintext phrase  :", phrase)

Private key m = 3835
Plaintext numbers : [1, 4, 1, 19, 20, 18, 1, 16, 5, 18, 1, 19, 16, 5, 18, 1]
Plaintext phrase  : ADASTRAPERASPERA
